In [1]:
#!/usr/bin/env python
# coding: utf-8
import pandas as pd
import numpy as np
from scipy.stats import wilcoxon


def formal_comparison(pair1, pair2):

    for method in pair1["method"].unique():
        x = np.array(pair1[pair1["method"] == method]["delta"])
        y = np.array(pair2[pair2["method"] == method]["delta"])
        print(method)
        print(wilcoxon(x, y))

def load_pair(treated_file, control_file, treated_col="blocks", control_col="noblocks"):
    df1 = pd.read_csv(treated_file).rename(columns={"score": treated_col})
    df2 = pd.read_csv(control_file).rename(columns={"score": control_col})
    pair = df1.merge(df2, on=["doc_id", "method", "membership"])
    pair["delta"] = pair[treated_col] - pair[control_col]
    pair = pair[pair["membership"] == "member"].copy()
    return pair


def report_att(df):
    print(df[["delta", "method"]].groupby(["method"]).mean().reset_index())



control_file = 'csvs/sutva_click2houston_com_2022-05-01_pair2_control_run4_sutva_click2houston_com_2022-05-01_pair2_control_run4_filtered.csv'
treated1 = 'csvs/sutva_click2houston_com_2022-05-01_pair1_treated_run1_sutva_click2houston_com_2022-05-01_pair2_control_run4_filtered.csv'
treated2 = 'csvs/sutva_click2houston_com_2022-05-01_pair2_treated_run3_sutva_click2houston_com_2022-05-01_pair2_control_run4_filtered.csv'

pair1 = load_pair(treated1, control_file)
pair2 = load_pair(treated2, control_file)

formal_comparison(pair1, pair2)

procd = pd.read_csv("data/processed/sutva_click2houston_com_2022-05-01_pair2_control_run4_filtered_aggregated_results.tsv", sep="\t")
procd = procd.rename(columns={"url": "doc_id"})

pair1 = pair1.merge(procd, on=["doc_id"])
pair1.columns = [o.replace('sutva_click2houston_com_2022-05-01_', '') for o in pair1.columns]
pair1["delta_count_median"] = pair1["pair1_treated_run1_median_count"] - pair1["pair2_control_run4_median_count"]
pair1["delta_count_mean"] = pair1["pair1_treated_run1_mean_count"] - pair1["pair2_control_run4_mean_count"]

pair2 = pair2.merge(procd,  on=["doc_id"])
pair2.columns = [o.replace('sutva_click2houston_com_2022-05-01_', '') for o in pair2.columns]
pair2["delta_count_median"] = pair1["pair2_treated_run3_median_count"] - pair1["pair2_control_run4_median_count"]
pair2["delta_count_mean"] = pair1["pair2_treated_run3_mean_count"] - pair1["pair2_control_run4_mean_count"]


report_att(pair1)
report_att(pair2)



loss
WilcoxonResult(statistic=32731.0, pvalue=0.0)
min_k
WilcoxonResult(statistic=160209.0, pvalue=0.0)
zlib
WilcoxonResult(statistic=64264.0, pvalue=0.0)
  method     delta
0   loss  1.716169
1  min_k  2.239866
2   zlib  0.001649
  method     delta
0   loss  2.075783
1  min_k  2.731475
2   zlib  0.001973


In [10]:
for method in ["loss"]:
    print(method)
    for p in [pair1, pair2]:
        T = p[p["method"] == method].copy()
        print(T[['delta_count_median', 'delta']].corr(method="kendall"))
        #print(T[['delta_count_mean', 'delta']].corr(method="kendall"))

loss
                    delta_count_median     delta
delta_count_median            1.000000 -0.124706
delta                        -0.124706  1.000000
                    delta_count_median     delta
delta_count_median            1.000000 -0.170892
delta                        -0.170892  1.000000


In [2]:
from scipy.stats import kendalltau
from scipy.stats import pearsonr
from scipy.stats import spearmanr


AGG = "median"

FIELD = f"delta_count_{AGG}"


ATEs = pair2[["doc_id", "delta", "method", "membership"]].rename(columns={"delta": "delta_p2"}).merge(pair1[["doc_id", "delta", "method", "membership"]], on=["doc_id", "method", "membership"])
ATEs["delta_ate"] = ATEs["delta_p2"] - ATEs["delta"]
ATEs = ATEs.drop(columns=["delta", "delta_p2"])

ATEs = ATEs[ATEs["method"] == "loss"].copy()

delta_counts = pair2[["doc_id", FIELD, "method", "membership"]].rename(columns={FIELD: f"{FIELD}_p2"}).merge(pair1[["doc_id", FIELD, "method", "membership"]], on=["doc_id", "method", "membership"])
delta_counts["delta_count"] = delta_counts[f"{FIELD}_p2"] - delta_counts[FIELD]
delta_counts = delta_counts.drop(columns=[f"{FIELD}_p2", FIELD])

ATEs = ATEs.merge(delta_counts, on = ['method', "membership", "doc_id"])

ATEs["abs_delta_ate"] = ATEs["delta_ate"].apply(lambda x: abs(x))
ATEs["abs_delta_count"] = ATEs["delta_count"].apply(lambda x: abs(x))

ATEs[["abs_delta_ate", "abs_delta_count"]].corr(method='kendall')

tau, pval = kendalltau(ATEs["abs_delta_ate"], ATEs["abs_delta_count"])

print(tau, pval)

0.0867402039204099 1.2674122640433688e-14


In [7]:
fn = "/Users/abha4861/dolma/data/interim/sutva/results/sutva_click2houston_com_2022-05-01_pair2_control_run4_filtered_in_sutva_click2houston_com_2022-05-01_pair2_control_run4_top1000_analysis_threshold0p9.csv"

card = pd.read_csv(fn)
card = card.rename(columns={"url": "doc_id"})

In [9]:
pair1 = pair1.merge(card, on='doc_id')
pair2 = pair2.merge(card, on='doc_id')

In [21]:
pair1[["delta", "sutva_click2houston_com_2022-05-01_pair1_treated_run1"]].corr(method="kendall")

,delta,sutva_click2houston_com_2022-05-01_pair1_treated_run1
delta,1.000000,-0.055465
sutva_click2houston_com_2022-05-01_pair1_treated_run1,-0.055465,1.000000


In [20]:
pair1[["pair1_treated_run1_mean_count", "sutva_click2houston_com_2022-05-01_pair1_treated_run1"]].corr(method="kendall")

,pair1_treated_run1_mean_count,sutva_click2houston_com_2022-05-01_pair1_treated_run1
pair1_treated_run1_mean_count,1.000000,0.134062
sutva_click2houston_com_2022-05-01_pair1_treated_run1,0.134062,1.000000
